# 3D U-Net + GAN Pipeline for Synthetic FDG-PET Generation

This notebook implements a deep learning pipeline for generating synthetic FDG-PET images from Dynamic Amyloid PET scans. The architecture combines a 3D U-Net with a Generative Adversarial Network (GAN) to progressively refine the quality of the synthetic volumes.

> **⚠️ PRIVACY & ETHICS NOTICE:**  
> This repository contains **code only**. No real patient data, DICOM files, or Protected Health Information (PHI) are included. All data processing scripts include robust, automated anonymization routines compliant with medical data privacy standards.

## 1. Import Required Libraries

In [ ]:
import os
import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pydicom
import nibabel as nib
from pydicom.pixel_data_handlers.util import apply_voi_lut
from pydicom.uid import generate_uid

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torchvision.utils import save_image

warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Data Anonymization & PHI Control
Ensures that all Protected Health Information (PHI), including institution names, patient names, and IDs, are permanently removed from DICOM headers before any processing.

In [ ]:
AMYLOID_PATH = "/kaggle/input/ano-dyn-ami-fmm"
FDG_PATH = "/kaggle/input/ano-fdg"
ANON_AMYLOID_PATH = "/kaggle/working/anon_amyloid"
ANON_FDG_PATH = "/kaggle/working/anon_fdg"

PHI_TAGS = [
    "PatientName", "PatientID", "PatientBirthDate", "PatientSex",
    "OtherPatientIDs", "OtherPatientNames", "InstitutionName",
    "ReferringPhysicianName", "InstitutionAddress", "StudyDescription"
]

def anonymize_dicom_file(in_path: str, out_path: str):
    """Removes ALL PHI and generates new UIDs for a single DICOM file."""
    ds = pydicom.dcmread(in_path)
    ds.PatientName = "ANONYMIZED"
    ds.PatientID = "000000"
    ds.PatientBirthDate = ""
    ds.PatientSex = ""
    ds.InstitutionName = ""
    ds.InstitutionAddress = ""
    ds.ReferringPhysicianName = ""
    ds.StudyDescription = ""
    ds.StudyInstanceUID = generate_uid()
    ds.SeriesInstanceUID = generate_uid()
    ds.SOPInstanceUID = generate_uid()
    ds.remove_private_tags()
    ds.save_as(out_path)

def anonymize_folder_silent(src_folder: str, dst_folder: str) -> int:
    """Iterates through a folder and anonymizes all DICOM files silently."""
    if os.path.exists(dst_folder):
        shutil.rmtree(dst_folder)
    os.makedirs(dst_folder)
    file_count = 0
    for root, _, files in os.walk(src_folder):
        for f in files:
            if f.endswith(".dcm"):
                src = os.path.join(root, f)
                dst = os.path.join(dst_folder, os.path.relpath(src, src_folder))
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                try:
                    anonymize_dicom_file(src, dst)
                    file_count += 1
                except Exception:
                    pass
    return file_count

def zip_folder(folder_path: str, zip_path: str):
    """Compresses a folder into a zip file."""
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for f in files:
                abs_path = os.path.join(root, f)
                rel_path = os.path.relpath(abs_path, folder_path)
                zipf.write(abs_path, rel_path)

# --- EXECUTION (Uncomment to run locally) ---
# print("Anonymizing Amyloid PET data...")
# amyloid_count = anonymize_folder_silent(AMYLOID_PATH, ANON_AMYLOID_PATH)
# print(f"Successfully anonymized {amyloid_count} Amyloid files.")
# zip_folder(ANON_AMYLOID_PATH, "/kaggle/working/anon_amyloid.zip")
#
# print("Anonymizing FDG-PET data...")
# fdg_count = anonymize_folder_silent(FDG_PATH, ANON_FDG_PATH)
# print(f"Successfully anonymized {fdg_count} FDG files.")
# zip_folder(ANON_FDG_PATH, "/kaggle/working/anon_fdg.zip")

## 3. Dataset Exploration
Validating the dataset structure without exposing sensitive patient identifiers.

In [ ]:
def explore_dataset_safely(amyloid_dir: str, fdg_dir: str):
    """Counts patients and total DICOM files without printing sensitive names."""
    try:
        amyloid_patients = [d for d in os.listdir(amyloid_dir) if os.path.isdir(os.path.join(amyloid_dir, d))]
        fdg_patients = [d for d in os.listdir(fdg_dir) if os.path.isdir(os.path.join(fdg_dir, d))]
        print(f"Dataset Overview:")
        print(f" - Amyloid PET Patient Directories: {len(amyloid_patients)}")
        print(f" - FDG-PET Patient Directories: {len(fdg_patients)}")
        
        amyloid_files = sum(len(files) for _, _, files in os.walk(amyloid_dir) if any(f.endswith('.dcm') for f in files))
        fdg_files = sum(len(files) for _, _, files in os.walk(fdg_dir) if any(f.endswith('.dcm') for f in files))
        print(f" - Total Amyloid DICOM files: {amyloid_files}")
        print(f" - Total FDG DICOM files: {fdg_files}")
    except FileNotFoundError:
        print("Dataset paths not found. Please update configuration.")

explore_dataset_safely(AMYLOID_PATH, FDG_PATH)

## 4. Dataset Class
Custom PyTorch Dataset for loading paired Amyloid and FDG PET volumes.

In [ ]:
class MedicalImageDataset(Dataset):
    def __init__(self, amyloid_paths, fdg_paths):
        self.amyloid_paths = amyloid_paths
        self.fdg_paths = fdg_paths

    def __len__(self):
        return len(self.amyloid_paths)

    def __getitem__(self, idx):
        amyloid_dcm = pydicom.dcmread(self.amyloid_paths[idx])
        amyloid_img = apply_voi_lut(amyloid_dcm.pixel_array, amyloid_dcm)
        
        fdg_dcm = pydicom.dcmread(self.fdg_paths[idx])
        fdg_img = apply_voi_lut(fdg_dcm.pixel_array, fdg_dcm)
        
        def normalize_image(image):
            img_min = image.min()
            img_max = image.max()
            if img_max - img_min == 0:
                return np.zeros_like(image)
            return (image - img_min) / (img_max - img_min)
        
        amyloid_img = normalize_image(amyloid_img)
        fdg_img = normalize_image(fdg_img)
        
        amyloid_tensor = torch.FloatTensor(amyloid_img).unsqueeze(0)
        fdg_tensor = torch.FloatTensor(fdg_img).unsqueeze(0)
        
        return amyloid_tensor, fdg_tensor

## 5. Model Architecture

```
DYNAMIC AMYLOID PET TO SYNTHETIC FDG-PET GENERATION PIPELINE

┌───────────────────────┐   ┌─────────────────┐   ┌─────────────────┐
│ DYNAMIC AMYLOID PET   ├───►│ 3D U-Net        ├───►│ GAN Refinement  │
│ (4D: x,y,z + time)    │   │ (Encoder-Decoder)│   │ (Adversarial)   │
└───────────────────────┘   └─────────────────┘   └────────┬────────┘
                                                           │
                                                           ▼
                                                  ┌─────────────────┐
                                                  │ SYNTHETIC FDG   │
                                                  │ (Output)        │
                                                  └─────────────────┘
```

In [ ]:
class UNet3D(nn.Module):
    def __init__(self, n_channels, n_classes):
        super(UNet3D, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        self.encoder1 = self.contracting_block(n_channels, 64)
        self.encoder2 = self.contracting_block(64, 128)
        self.encoder3 = self.contracting_block(128, 256)
        self.encoder4 = self.contracting_block(256, 512)
        
        self.bottleneck = nn.Sequential(
            nn.Conv3d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm3d(1024),
            nn.ReLU(inplace=True),
            nn.Conv3d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm3d(1024),
            nn.ReLU(inplace=True)
        )
        
        self.upconv4 = nn.ConvTranspose3d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = self.contracting_block(1024, 512)
        self.upconv3 = nn.ConvTranspose3d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = self.contracting_block(512, 256)
        self.upconv2 = nn.ConvTranspose3d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = self.contracting_block(256, 128)
        self.upconv1 = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = self.contracting_block(128, 64)
        
        self.final = nn.Conv3d(64, n_classes, kernel_size=1)

    def contracting_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool3d(enc1, 2))
        enc3 = self.encoder3(F.max_pool3d(enc2, 2))
        enc4 = self.encoder4(F.max_pool3d(enc3, 2))
        
        bottleneck = self.bottleneck(F.max_pool3d(enc4, 2))
        
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.decoder1(dec1)
        
        return self.final(dec1)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv3d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(512, 1, kernel_size=4, stride=1, padding=0),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

## 6. Training Pipeline
GAN training loop with adversarial and pixel-wise losses.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LR = 0.0002
B1 = 0.5
B2 = 0.999
N_EPOCHS = 100
SAMPLE_INTERVAL = 400

def train_gan(generator, discriminator, train_loader, n_epochs=100):
    adversarial_loss = nn.BCELoss()
    pixelwise_loss = nn.L1Loss()
    
    optimizer_G = torch.optim.Adam(generator.parameters(), lr=LR, betas=(B1, B2))
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(B1, B2))
    
    for epoch in range(n_epochs):
        for i, (imgs, labels) in enumerate(train_loader):
            valid = torch.ones((imgs.size(0), 1, 1, 1, 1), requires_grad=False).to(DEVICE)
            fake = torch.zeros((imgs.size(0), 1, 1, 1, 1), requires_grad=False).to(DEVICE)
            
            real_imgs = imgs.to(DEVICE)
            real_labels = labels.to(DEVICE)
            
            # Generator
            optimizer_G.zero_grad()
            gen_imgs = generator(real_imgs)
            g_loss = 0.001 * adversarial_loss(discriminator(gen_imgs), valid) + 0.999 * pixelwise_loss(gen_imgs, real_labels)
            g_loss.backward()
            optimizer_G.step()
            
            # Discriminator
            optimizer_D.zero_grad()
            real_loss = adversarial_loss(discriminator(real_labels), valid)
            fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
            d_loss = (real_loss + fake_loss) / 2
            d_loss.backward()
            optimizer_D.step()
            
            batches_done = epoch * len(train_loader) + i
            if batches_done % SAMPLE_INTERVAL == 0:
                print(f"[Epoch {epoch}/{n_epochs}] [Batch {i}/{len(train_loader)}] [D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")
    
    return generator, discriminator

## 7. Evaluation & Visualization

In [ ]:
def visualize_results(generator, test_loader, num_samples=4):
    generator.eval()
    with torch.no_grad():
        test_input, _ = next(iter(test_loader))
        test_input = test_input[:num_samples].to(DEVICE)
        synthetic_fdg = generator(test_input)
        
        fig, axes = plt.subplots(2, num_samples, figsize=(15, 8))
        for i in range(num_samples):
            axes[0, i].imshow(test_input[i, 0].cpu().numpy().squeeze(), cmap='gray')
            axes[0, i].set_title(f'Amyloid PET {i+1}')
            axes[0, i].axis('off')
            
            axes[1, i].imshow(synthetic_fdg[i, 0].cpu().numpy().squeeze(), cmap='gray')
            axes[1, i].set_title(f'Synthetic FDG-PET {i+1}')
            axes[1, i].axis('off')
        
        plt.tight_layout()
        plt.show()

# --- EXECUTION (Uncomment to run) ---
# amyloid_files = [os.path.join(r, f) for r, _, fs in os.walk(ANON_AMYLOID_PATH) for f in fs if f.endswith('.dcm')]
# fdg_files = [os.path.join(r, f) for r, _, fs in os.walk(ANON_FDG_PATH) for f in fs if f.endswith('.dcm')]
# 
# train_files, test_files = train_test_split(list(zip(amyloid_files, fdg_files)), test_size=0.2, random_state=42)
# train_amyloid, train_fdg = zip(*train_files)
# test_amyloid, test_fdg = zip(*test_files)
# 
# train_dataset = MedicalImageDataset(train_amyloid, train_fdg)
# test_dataset = MedicalImageDataset(test_amyloid, test_fdg)
# 
# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=0)
# 
# generator = UNet3D(n_channels=1, n_classes=1).to(DEVICE)
# discriminator = Discriminator().to(DEVICE)
# 
# generator, discriminator = train_gan(generator, discriminator, train_loader, n_epochs=N_EPOCHS)
# visualize_results(generator, test_loader)